
## 0. Environment Setup
Detect runtime environment (Colab or local) and mount Drive if needed.

In [1]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running in Google Colab — Drive mounted.')
else:
    print('Running locally (VS Code / Jupyter).')

Mounted at /content/drive
Running in Google Colab — Drive mounted.


In [2]:
# Install required packages (run once) , (skip on Colab — already available)
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pillow", "matplotlib", "numpy",
                       "torch", "torchvision", "scikit-learn"])

0

## 1. Import libraries and set dataset path
We import all required libraries upfront and specify the dataset path.

In [3]:
import os
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [6]:
import os, subprocess

if IN_COLAB:
    local_base = '/content/xai_dataset'

    if os.path.exists(f'{local_base}/CUB_200_2011'):
        print("Dataset already exists — skipping download.")
    else:
        os.makedirs(local_base, exist_ok=True)
        print("Downloading CUB-200-2011 (~1.2GB)...")
        subprocess.run(['wget', '-q', '--show-progress',
            'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1',
            '-O', '/content/CUB_200_2011.tgz'], check=True)
        subprocess.run(['tar', '-xzf', '/content/CUB_200_2011.tgz',
                        '-C', local_base], check=True)
        os.remove('/content/CUB_200_2011.tgz')
        print("CUB-200-2011 extracted.")

        print("Downloading segmentation masks (~39MB)...")
        subprocess.run(['wget', '-q', '--show-progress',
            'https://data.caltech.edu/records/w9d68-gec53/files/segmentations.tgz?download=1',
            '-O', '/content/segmentations.tgz'], check=True)
        subprocess.run(['tar', '-xzf', '/content/segmentations.tgz',
                        '-C', local_base], check=True)
        os.remove('/content/segmentations.tgz')
        print("Masks extracted.")

    # Override paths to use local SSD instead of Drive
    dataset_dir = f'{local_base}/CUB_200_2011/'
    attr_txt    = f'{local_base}/attributes.txt'
    mask_root   = f'{local_base}/segmentations/'

    n_imgs  = sum(1 for _,_,fs in os.walk(dataset_dir) for f in fs if f.endswith('.jpg'))
    n_masks = sum(1 for _,_,fs in os.walk(mask_root)   for f in fs if f.endswith('.png'))
    print(f"\nImages  : {n_imgs}")
    print(f"Masks   : {n_masks}")
    print(f"dataset_dir : {dataset_dir}")
else:
    print("Local environment — using DB/ paths defined above.")

Dataset already exists — skipping download.

Images  : 11788
Masks   : 11788
dataset_dir : /content/xai_dataset/CUB_200_2011/


## 2. Load Image List and Class Labels
reads images.txt and image_class_labels.txt — builds the master list of 11,788 image paths and their species labels (1-indexed, 1–200).

In [7]:
# Read image paths
with open(os.path.join(dataset_dir, 'images.txt')) as f:
    image_list = [line.strip().split(' ')[1] for line in f]

# Read class labels (1-indexed: 1–200)
with open(os.path.join(dataset_dir, 'image_class_labels.txt')) as f:
    labels = [int(line.strip().split(' ')[1]) for line in f]

print(f'Images : {len(image_list)}')
print(f'Labels : {len(labels)}')
print(f'Example: {image_list[0]} → class {labels[0]}')

Images : 11788
Labels : 11788
Example: 001.Black_footed_Albatross/Black_Footed_Albatross_0046_18.jpg → class 1


 3 — آپدیت شده:

## 3. Load Attribute Names and Per-Image Annotations
Reads 312 attribute names and per-image attribute labels with certainty scores. Only attributes with certainty >= 3 will be trusted during loss computation.

In [8]:
# attr_txt is already set in the paths cell above
with open(attr_txt) as f:
    attr_names = [line.strip().split(' ', 1)[1] for line in f]

print(f'Attributes: {len(attr_names)}')
print(f'Example: {attr_names[0]}')

# Format per line: img_id  attr_id  is_present  certainty  time
attr_label_txt = os.path.join(dataset_dir, 'attributes', 'image_attribute_labels.txt')
image_attr = {}
image_attr_certainty = {}

with open(attr_label_txt) as f:
    for line in f:
        parts = line.strip().split()
        img_id    = int(parts[0])
        attr_id   = int(parts[1]) - 1  # 0-indexed
        is_present = int(parts[2])
        certainty  = int(parts[3])
        image_attr.setdefault(img_id, {})[attr_id] = is_present
        image_attr_certainty.setdefault(img_id, {})[attr_id] = certainty

print(f'Images with attributes: {len(image_attr)}')


Attributes: 312
Example: has_bill_shape::curved_(up_or_down)
Images with attributes: 11788


## 4. Build attribute vectors and certainty masks
Convert sparse per-image attribute dicts to dense arrays.
Also build certainty masks: 1 where certainty >= 3 (trusted), 0 otherwise.

In [9]:
num_attrs = len(attr_names)
attr_vectors = []
certainty_masks = []

for i in range(1, len(image_list) + 1):
    vec  = np.zeros(num_attrs, dtype=np.int8)
    cert = np.zeros(num_attrs, dtype=np.int8)
    for attr_id, is_present in image_attr.get(i, {}).items():
        vec[attr_id]  = is_present
        cert[attr_id] = 1 if image_attr_certainty.get(i, {}).get(attr_id, 0) >= 3 else 0
    attr_vectors.append(vec)
    certainty_masks.append(cert)

print(f'Attribute vectors : {len(attr_vectors)}')
print(f'Certainty masks   : {len(certainty_masks)}')
print(f'Avg trusted attrs : {np.mean([c.sum() for c in certainty_masks]):.1f} / 312')
print(f'Example vec  (img1): {attr_vectors[0].sum()} attributes present')
print(f'Example cert (img1): {certainty_masks[0].sum()} attributes trusted')

Attribute vectors : 11788
Certainty masks   : 11788
Avg trusted attrs : 259.2 / 312
Example vec  (img1): 12 attributes present
Example cert (img1): 127 attributes trusted


## 5. Split data into train, val, and test
We use `train_test_split.txt` for the official CUB train/test split, then carve out 10% of train as a stratified validation set.

In [10]:
split_txt = os.path.join(dataset_dir, 'train_test_split.txt')
with open(split_txt) as f:
    is_train = [int(line.strip().split(' ')[1]) for line in f]

image_ids = list(range(1, len(image_list) + 1))

all_train_images = [img for img, t in zip(image_list,     is_train) if t == 1]
all_train_labels = [lbl for lbl, t in zip(labels,         is_train) if t == 1]
all_train_attrs  = [vec for vec, t in zip(attr_vectors,   is_train) if t == 1]
all_train_certs  = [c   for c,   t in zip(certainty_masks,is_train) if t == 1]
all_train_ids    = [iid for iid, t in zip(image_ids,      is_train) if t == 1]

test_images = [img for img, t in zip(image_list,     is_train) if t == 0]
test_labels = [lbl for lbl, t in zip(labels,         is_train) if t == 0]
test_attrs  = [vec for vec, t in zip(attr_vectors,   is_train) if t == 0]
test_certs  = [c   for c,   t in zip(certainty_masks,is_train) if t == 0]
test_ids    = [iid for iid, t in zip(image_ids,      is_train) if t == 0]

indices = list(range(len(all_train_images)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.1, random_state=42,
    stratify=all_train_labels
)

train_images = [all_train_images[i] for i in train_idx]
train_labels = [all_train_labels[i] for i in train_idx]
train_attrs  = [all_train_attrs[i]  for i in train_idx]
train_certs  = [all_train_certs[i]  for i in train_idx]
train_ids    = [all_train_ids[i]    for i in train_idx]

val_images   = [all_train_images[i] for i in val_idx]
val_labels   = [all_train_labels[i] for i in val_idx]
val_attrs    = [all_train_attrs[i]  for i in val_idx]
val_certs    = [all_train_certs[i]  for i in val_idx]
val_ids      = [all_train_ids[i]    for i in val_idx]

print(f'Train : {len(train_images)}')
print(f'Val   : {len(val_images)}')
print(f'Test  : {len(test_images)}')

Train : 5394
Val   : 600
Test  : 5794


## 6. Parse L1 concept groups from attribute names
Build the L1 (coarse) concept hierarchy by parsing `attributes.txt` names.
Format: `has_{part}_{property}::{value}` → extract `{part}` as L1 group.

Also load Part Locations (`part_locs.txt`) for **visibility masking** — parts not visible in an image should be masked in L1 loss.

Attributes without a clear body-part parent (`primary_color`, `size`, `shape`) are assigned `parent_idx = -1` (always unmasked).

In [11]:
# --- Step 1: Parse attribute names → extract part groups ---
MERGE_MAP = {
    'under_tail': 'tail',
    'upper_tail': 'tail',
    'upperparts': 'back',
    'underparts': 'belly',
}
NO_PARENT = {'primary', 'shape', 'size'}

part_to_attr_indices = {}
attr_to_part = {}

for idx, name in enumerate(attr_names):
    # e.g. 'has_bill_shape::dagger' → category = 'bill_shape'
    category = name.split('::')[0].replace('has_', '')
    # strip property suffix → part name
    for suffix in ('_color', '_pattern', '_shape', '_length'):
        if category.endswith(suffix):
            part = category[:len(category) - len(suffix)]
            break
    else:
        part = category

    # merge compound parts
    part = MERGE_MAP.get(part, part)

    if part in NO_PARENT:
        attr_to_part[idx] = None
        continue

    attr_to_part[idx] = part
    part_to_attr_indices.setdefault(part, []).append(idx)

CONCEPT_NAMES = sorted(part_to_attr_indices.keys())
NUM_L1 = len(CONCEPT_NAMES)
NUM_L2 = len(attr_names)
part_to_idx = {p: i for i, p in enumerate(CONCEPT_NAMES)}

# Build parent index for each L2 attribute (for Masked Fine Head)
# -1 means "no parent" (always unmasked)
attr_parent_idx = []
for idx in range(NUM_L2):
    part = attr_to_part.get(idx)
    attr_parent_idx.append(part_to_idx[part] if part is not None else -1)
attr_parent_idx = np.array(attr_parent_idx, dtype=np.int32)

print(f'L1 concept groups ({NUM_L1}): {CONCEPT_NAMES}')
for p in CONCEPT_NAMES:
    print(f'  {p:12s} → {len(part_to_attr_indices[p]):3d} attributes')
no_parent_count = sum(1 for v in attr_to_part.values() if v is None)
print(f'Attributes with no L1 parent: {no_parent_count}')

# --- Step 2: Load Part Locations for visibility masking ---
# parts.txt: part_id part_name — 15 CUB parts
# part_locs.txt: img_id part_id x y visible
CUB_PART_TO_L1 = {
    1: 'back',    2: 'bill',    3: 'belly',   4: 'breast',
    5: 'crown',   6: 'forehead', 7: 'eye',    8: 'leg',
    9: 'wing',   10: 'nape',   11: 'eye',    12: 'leg',
    13: 'wing',  14: 'tail',   15: 'throat',
}

part_locs_txt = os.path.join(dataset_dir, 'parts', 'part_locs.txt')
image_part_visible = {}
with open(part_locs_txt) as f:
    for line in f:
        cols = line.strip().split()
        img_id = int(cols[0])
        cub_part_id = int(cols[1])
        visible = int(cols[4])
        l1_part = CUB_PART_TO_L1.get(cub_part_id)
        if l1_part and l1_part in part_to_idx:
            image_part_visible.setdefault(img_id, {})
            l1_idx = part_to_idx[l1_part]
            # OR: visible if EITHER left or right is visible
            image_part_visible[img_id][l1_idx] = max(
                image_part_visible[img_id].get(l1_idx, 0), visible
            )

# "head" has no direct CUB part — mark visible if any of crown/forehead/eye/nape are visible
head_subparts = ['crown', 'forehead', 'eye', 'nape']
if 'head' in part_to_idx:
    head_idx = part_to_idx['head']
    sub_idxs = [part_to_idx[s] for s in head_subparts if s in part_to_idx]
    for img_id in image_part_visible:
        image_part_visible[img_id][head_idx] = max(
            image_part_visible[img_id].get(si, 0) for si in sub_idxs
        )

print(f'Images with visibility data: {len(image_part_visible)}')

L1 concept groups (13): ['back', 'belly', 'bill', 'breast', 'crown', 'eye', 'forehead', 'head', 'leg', 'nape', 'tail', 'throat', 'wing']
  back         →  34 attributes
  belly        →  34 attributes
  bill         →  27 attributes
  breast       →  19 attributes
  crown        →  15 attributes
  eye          →  14 attributes
  forehead     →  15 attributes
  head         →  11 attributes
  leg          →  15 attributes
  nape         →  15 attributes
  tail         →  40 attributes
  throat       →  15 attributes
  wing         →  24 attributes
Attributes with no L1 parent: 34
Images with visibility data: 11788


## 6b. Build L1, L2, certainty, and visibility arrays per split
Compute final numpy arrays for each split using the parsed L1 groups, certainty masks, and part visibility data.

In [12]:
def build_l1_labels(attr_vecs):
    """OR-aggregation: L1[part] = 1 if ANY child attribute == 1."""
    result = np.zeros((len(attr_vecs), NUM_L1), dtype=np.float32)
    for i, vec in enumerate(attr_vecs):
        for part, idxs in part_to_attr_indices.items():
            if any(vec[j] for j in idxs):
                result[i, part_to_idx[part]] = 1.0
    return result

def build_visibility(img_id_list):
    """Build (N, NUM_L1) visibility matrix from part_locs data."""
    result = np.zeros((len(img_id_list), NUM_L1), dtype=np.float32)
    for i, iid in enumerate(img_id_list):
        vis = image_part_visible.get(iid, {})
        for l1_idx, v in vis.items():
            result[i, l1_idx] = float(v)
    return result

# --- Train ---
train_l1              = build_l1_labels(train_attrs)
train_l2              = np.array(train_attrs, dtype=np.float32)
train_certainty_mask  = np.array(train_certs, dtype=np.float32)
train_visibility      = build_visibility(train_ids)

# --- Val ---
val_l1                = build_l1_labels(val_attrs)
val_l2                = np.array(val_attrs, dtype=np.float32)
val_certainty_mask    = np.array(val_certs, dtype=np.float32)
val_visibility        = build_visibility(val_ids)

# --- Test ---
test_l1               = build_l1_labels(test_attrs)
test_l2               = np.array(test_attrs, dtype=np.float32)
test_certainty_mask   = np.array(test_certs, dtype=np.float32)
test_visibility       = build_visibility(test_ids)

print(f'Train — L1: {train_l1.shape}, L2: {train_l2.shape}, cert: {train_certainty_mask.shape}, vis: {train_visibility.shape}')
print(f'Val   — L1: {val_l1.shape}, L2: {val_l2.shape}, cert: {val_certainty_mask.shape}, vis: {val_visibility.shape}')
print(f'Test  — L1: {test_l1.shape}, L2: {test_l2.shape}, cert: {test_certainty_mask.shape}, vis: {test_visibility.shape}')
print(f'\nL1 concept names: {CONCEPT_NAMES}')
print(f'attr_parent_idx sample (first 20): {attr_parent_idx[:20]}')

Train — L1: (5394, 13), L2: (5394, 312), cert: (5394, 312), vis: (5394, 13)
Val   — L1: (600, 13), L2: (600, 312), cert: (600, 312), vis: (600, 13)
Test  — L1: (5794, 13), L2: (5794, 312), cert: (5794, 312), vis: (5794, 13)

L1 concept names: ['back', 'belly', 'bill', 'breast', 'crown', 'eye', 'forehead', 'head', 'leg', 'nape', 'tail', 'throat', 'wing']
attr_parent_idx sample (first 20): [ 2  2  2  2  2  2  2  2  2 12 12 12 12 12 12 12 12 12 12 12]


## 7. Load Segmentation Masks
Load the binary segmentation masks from DB1-Mask. Each mask marks which pixels belong to the bird (1) vs background (0).

In [13]:
# mask_root is already set in the paths cell above
mask_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

mask_count = sum(
    1 for root, _, files in os.walk(mask_root)
    for f in files if f.endswith('.png')
)
print(f'Mask root: {mask_root}')
print(f'Total mask files found: {mask_count}')


Mask root: /content/xai_dataset/segmentations/
Total mask files found: 11788


## 8. PyTorch Dataset and DataLoader
Wraps images, labels, concept annotations, certainty masks, visibility masks, and segmentation masks into a unified `BirdDataset`.

Each batch returns 7 items: `(imgs, lbls, l1s, l2s, masks, certs, viss)`.

In [14]:
IMG_SIZE   = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class BirdDataset(Dataset):
    def __init__(self, image_paths, labels, l1_labels, l2_labels,
                 certainty_mask, visibility,
                 img_root, mask_root, transform=None, mask_transform=None):
        self.paths          = image_paths
        self.labels         = labels
        self.l1             = l1_labels          # (N, NUM_L1)
        self.l2             = l2_labels          # (N, 312)
        self.cert           = certainty_mask     # (N, 312)
        self.vis            = visibility          # (N, NUM_L1)
        self.img_root       = img_root
        self.mask_root      = mask_root
        self.transform      = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        rel_path = self.paths[idx]

        # --- image ---
        img = Image.open(os.path.join(self.img_root, 'images', rel_path)).convert('RGB')
        if self.transform:
            img = self.transform(img)

        # --- segmentation mask ---
        mask_path = os.path.join(self.mask_root,
                                 os.path.splitext(rel_path)[0] + '.png')
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('L')
            if self.mask_transform:
                mask = self.mask_transform(mask)
        else:
            mask = torch.zeros(1, IMG_SIZE, IMG_SIZE)

        # --- labels ---
        label = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        l1    = torch.tensor(self.l1[idx],   dtype=torch.float32)
        l2    = torch.tensor(self.l2[idx],   dtype=torch.float32)
        cert  = torch.tensor(self.cert[idx], dtype=torch.float32)
        vis   = torch.tensor(self.vis[idx],  dtype=torch.float32)

        return img, label, l1, l2, mask, cert, vis


# --- Build datasets ---
train_dataset = BirdDataset(
    train_images, train_labels, train_l1, train_l2,
    train_certainty_mask, train_visibility,
    dataset_dir, mask_root, train_transform, mask_transform
)
val_dataset = BirdDataset(
    val_images, val_labels, val_l1, val_l2,
    val_certainty_mask, val_visibility,
    dataset_dir, mask_root, eval_transform, mask_transform
)
test_dataset = BirdDataset(
    test_images, test_labels, test_l1, test_l2,
    test_certainty_mask, test_visibility,
    dataset_dir, mask_root, eval_transform, mask_transform
)

# --- Build dataloaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Verify one batch
imgs, lbls, l1s, l2s, masks, certs, viss = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  imgs:  {imgs.shape}')     # (32, 3, 224, 224)
print(f'  lbls:  {lbls.shape}')     # (32,)
print(f'  l1s:   {l1s.shape}')      # (32, NUM_L1)
print(f'  l2s:   {l2s.shape}')      # (32, 312)
print(f'  masks: {masks.shape}')    # (32, 1, 224, 224)
print(f'  certs: {certs.shape}')    # (32, 312)
print(f'  viss:  {viss.shape}')     # (32, NUM_L1)

Train batches: 169
Val batches:   19
Test batches:  182


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Batch shapes:
  imgs:  torch.Size([32, 3, 224, 224])
  lbls:  torch.Size([32])
  l1s:   torch.Size([32, 13])
  l2s:   torch.Size([32, 312])
  masks: torch.Size([32, 1, 224, 224])
  certs: torch.Size([32, 312])
  viss:  torch.Size([32, 13])


## 9. Save Pipeline Metadata
Save model configuration and hierarchy mapping to Google Drive for use in model.ipynb and train.ipynb.

In [18]:
import pickle
import numpy as np

output_dir = local_base  # /content/xai_dataset
save_path = os.path.join(output_dir, 'data_pipeline.pkl')

save_dict = {
    'NUM_L1':          NUM_L1,
    'NUM_L2':          NUM_L2,
    'CONCEPT_NAMES':   CONCEPT_NAMES,
    'attr_parent_idx': attr_parent_idx,
}

with open(save_path, 'wb') as f:
    pickle.dump(save_dict, f)

print(f'Saved to {save_path}')
print(f'NUM_L1: {NUM_L1}')
print(f'NUM_L2: {NUM_L2}')
print(f'CONCEPT_NAMES: {CONCEPT_NAMES}')
print(f'attr_parent_idx shape: {attr_parent_idx.shape}')


Saved to /content/xai_dataset/data_pipeline.pkl
NUM_L1: 13
NUM_L2: 312
CONCEPT_NAMES: ['back', 'belly', 'bill', 'breast', 'crown', 'eye', 'forehead', 'head', 'leg', 'nape', 'tail', 'throat', 'wing']
attr_parent_idx shape: (312,)
